# Tea Flush-Stage Classifier — Transfer Learning

**Workstream B · TINAT ML Contribution — Angel Bohorquez, contributor**

---

## Objective

Train a lightweight image classifier that maps a tea-plant photo to one of three
crop stages (the **WHEN** dimension of the WHO×WHERE×WHEN metric):

| Class idx | Label | Months | Risk |
|-----------|-------|--------|------|
| 0 | `dormant` | 0–2 | Low |
| 1 | `new_flush` | 3–4 | **HIGH** — prime Helopeltis window |
| 2 | `mature_leaf` | 5+ | Low |

## Architecture

**MobileNetV2** (ImageNet pre-trained) with a custom 3-class head.  
~3.4M params, fast CPU inference, clean TFLite export.

## Data needed — collect before running §4 onwards

Place images here — one subfolder per class:
```
data/raw/flush_stages/
    dormant/          ← tea plant with no new shoots (dark, mature)
    new_flush/        ← tender 2–3 leaves + bud (bright green, soft)
    mature_leaf/      ← fully expanded dark leaves
```
Aim for **≥ 150 images per class** before training.  
Sources: iNaturalist (*Camellia sinensis*), field photos labelled by harvest month.

## 1. Environment

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

ROOT     = Path("..").resolve()
sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw" / "flush_stages"
MODELS   = ROOT / "models"
CLASSES  = ["dormant", "new_flush", "mature_leaf"]

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Classes  : {CLASSES}")
print(f"Data dir : {DATA_RAW}")

## 2. Data check

Run this anytime to see progress — you don't need all images before running it.

In [ ]:
total = 0
ready = True

for cls in CLASSES:
    folder = DATA_RAW / cls
    if not folder.exists():
        print(f"  {cls:15s}: folder missing — create data/raw/flush_stages/{cls}/")
        ready = False
        continue
    imgs = list(folder.glob("*.jpg")) + list(folder.glob("*.png")) + list(folder.glob("*.jpeg"))
    status = "OK" if len(imgs) >= 150 else f"need {150 - len(imgs)} more"
    print(f"  {cls:15s}: {len(imgs):4d} images  [{status}]")
    total += len(imgs)
    if len(imgs) < 150:
        ready = False

print(f"\nTotal: {total} images")
print(f"Ready to train: {ready}")

## 3. Train / val / test split

Run **once** after images are collected.  
Creates `data/raw/flush_stages/{train,val,test}/` with class subfolders inside.

In [ ]:
# ── Run once when ≥ 150 images/class are ready ───────────────────────────────
# import shutil
# from sklearn.model_selection import train_test_split
#
# for cls in CLASSES:
#     imgs = list((DATA_RAW / cls).glob("*.*"))
#     train, test = train_test_split(imgs, test_size=0.15, random_state=42)
#     train, val  = train_test_split(train, test_size=0.15/0.85, random_state=42)
#     for split_name, split_imgs in [("train", train), ("val", val), ("test", test)]:
#         dest = DATA_RAW / split_name / cls
#         dest.mkdir(parents=True, exist_ok=True)
#         for img in split_imgs:
#             shutil.copy(img, dest / img.name)
#     print(f"{cls}: train={len(train)} val={len(val)} test={len(test)}")

print("Split cell ready — uncomment when ≥ 150 images/class are collected.")

## 4. Transforms

Training: aggressive colour jitter + rotation to simulate field light variation.  
Validation: deterministic centre crop.

In [ ]:
IMG_SIZE = 224

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("Transforms defined.")

## 5. DataLoaders

In [ ]:
# ── Uncomment after running the split cell (§3) ───────────────────────────────
# train_ds = datasets.ImageFolder(str(DATA_RAW / "train"), transform=train_tf)
# val_ds   = datasets.ImageFolder(str(DATA_RAW / "val"),   transform=val_tf)
# test_ds  = datasets.ImageFolder(str(DATA_RAW / "test"),  transform=val_tf)
#
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
# val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=0)
# test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
#
# print(f"Train : {len(train_ds)} samples")
# print(f"Val   : {len(val_ds)} samples")
# print(f"Test  : {len(test_ds)} samples")
# print(f"Class order: {train_ds.class_to_idx}")
# # Must match CLASSES = ['dormant', 'new_flush', 'mature_leaf']

print("DataLoader cell ready — uncomment after split.")

## 6. Model — MobileNetV2 with custom head

In [ ]:
def build_model(num_classes: int = 3, freeze_backbone: bool = True) -> nn.Module:
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


model = build_model(num_classes=3, freeze_backbone=True)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}  (head only — backbone frozen)")

## 7. Training

- **Phase 1 (epochs 1–5):** backbone frozen, only the 3-class head trains — fast convergence
- **Phase 2 (epochs 6–20):** all layers unfreeze, low LR — fine-tuning

In [ ]:
# ── Uncomment after DataLoaders are ready ────────────────────────────────────
# device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model     = model.to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(
#     filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3
# )
#
# PHASE1, PHASE2 = 5, 15
# history = {"train_loss": [], "val_acc": []}
#
# def run_epoch(loader, train=True):
#     model.train(train)
#     total_loss, correct, n = 0, 0, 0
#     with torch.set_grad_enabled(train):
#         for x, y in loader:
#             x, y = x.to(device), y.to(device)
#             logits = model(x)
#             loss   = criterion(logits, y)
#             if train:
#                 optimizer.zero_grad(); loss.backward(); optimizer.step()
#             total_loss += loss.item() * len(y)
#             correct    += (logits.argmax(1) == y).sum().item()
#             n          += len(y)
#     return total_loss / n, correct / n
#
# for epoch in range(1, PHASE1 + PHASE2 + 1):
#     if epoch == PHASE1 + 1:
#         for param in model.parameters(): param.requires_grad = True
#         optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
#         print("Phase 2: full fine-tuning")
#     tr_loss, _ = run_epoch(train_loader, train=True)
#     _, val_acc = run_epoch(val_loader,   train=False)
#     history["train_loss"].append(tr_loss)
#     history["val_acc"].append(val_acc)
#     print(f"Epoch {epoch:3d} | loss {tr_loss:.4f} | val_acc {val_acc:.3f}")

print("Training cell ready — uncomment after DataLoaders are set up.")

## 8. Evaluation

Target: val accuracy ≥ 85% on `new_flush` — missing a high-risk stage is costly.

In [ ]:
# ── Uncomment after training ──────────────────────────────────────────────────
# from sklearn.metrics import classification_report, confusion_matrix
# import seaborn as sns
# import matplotlib.pyplot as plt
#
# model.eval()
# all_preds, all_labels = [], []
# with torch.no_grad():
#     for x, y in test_loader:
#         preds = model(x.to(device)).argmax(1).cpu()
#         all_preds.extend(preds.numpy())
#         all_labels.extend(y.numpy())
#
# print(classification_report(all_labels, all_preds, target_names=CLASSES))
#
# cm = confusion_matrix(all_labels, all_preds)
# sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASSES, yticklabels=CLASSES)
# plt.title("Flush-Stage Classifier — Confusion Matrix")
# plt.tight_layout()
# plt.show()

print("Evaluation cell ready — uncomment after training.")

## 9. Save and export to ONNX

The ONNX file gets picked up automatically by `fusion.py`'s `FlushClassifier` once converted to TFLite.

In [ ]:
# ── Uncomment after training ──────────────────────────────────────────────────
# import torch.onnx
#
# MODELS.mkdir(exist_ok=True)
# torch.save(model.state_dict(), MODELS / "flush_stage_mobilenetv2.pt")
# print("PyTorch checkpoint saved.")
#
# model.cpu().eval()
# dummy = torch.randn(1, 3, 224, 224)
# torch.onnx.export(
#     model, dummy,
#     MODELS / "flush_stage.onnx",
#     input_names=["image"],
#     output_names=["logits"],
#     opset_version=17,
# )
# print("ONNX exported → models/flush_stage.onnx")
# print("Next: convert with:  python -m tf2onnx.convert --onnx models/flush_stage.onnx --output models/flush_stage.tflite --target tflite")

print("Save / export cell ready — uncomment after training.")

## Next steps

1. Collect ≥ 150 images per class → `data/raw/flush_stages/{dormant,new_flush,mature_leaf}/`
2. Run §3 split → §5 DataLoaders → §7 Training → §8 Evaluation
3. Export ONNX (§9) → convert to TFLite
4. Place `flush_stage.tflite` in `models/`
5. `fusion.py` will pick it up automatically via `FlushClassifier`